# Enunciado práctica.

### Recursos

Problemas interesantes para Aprendizaje por refuerzo
 * Gymnasium: https://gymnasium.farama.org/environments/box2d/
 * Solución del Lunar Lander con DQN: https://shiva-verma.medium.com/solving-lunar-lander-openaigym-reinforcement-learning-785675066197
 * Otra solución: https://wingedsheep.com/lunar-lander-dqn/ y https://github.com/wingedsheep/blog/blob/main/lunar-lander-dqn/lunar_lander_dqn_blog.ipynb
 * The Nature of Code: https://youtu.be/lu5ul7z4icQ
 * Librería para neuroevolución: https://pypi.org/project/nevopy/

## Instalación

%pip install gymnasium  
%pip install gymnasium[box2d] 

### Acciones adicionales

#### En macos

pip uninstall swig  
xcode-select -—install (si no se tienen ya)  
pip install swig  / sudo port install swig-python
pip install 'gymnasium[box2d]' # en zsh hay que poner las comillas  

Si se da el error [NSCheapMutableString initialize] may have been in progress in another thread when fork() was called. We cannot safely call it or ignore it in the fork() child process. Crashing instead.  
Hacer  
OBJC_DISABLE_INITIALIZE_FORK_SAFETY=YES

#### en Windows

Si da error, se debe a la falta de la versión correcta de Microsoft Visual C++ Build Tools, que es una dependencia de Box2D. Para solucionar este problema, puede seguir los siguientes pasos:  
 * Descargar Microsoft Visual C++ Build Tools desde https://visualstudio.microsoft.com/visual-cpp-build-tools/.
 * Dentro de la app, seleccione la opción "Herramientas de compilación de C++" para instalar.
 * Reinicie su sesión en Jupyter Notebook.
 * Ejecute nuevamente el comando !pip install gymnasium[box2d] en la línea de comandos de su notebook.
 * A algunos les ha funcioando instalando antes pygame.

También puede deberse a no tener swig instalado o a tener una versión incorrecta, en ese caso habrá que instalarlo con

%pip uninstall swig  
%pip install swig

Si nada funciona, la solución última es instalar anaconda y cambiar el kernel.

In [1]:
!pip install -r ../../requirements.txt --quiet

<h1 style='color:red ; text-align:center'>Apartado A</h1>

# Prueba alumno.

grabamos varias partidas jugando con el teclado y escribimos nuestras conclusiones después de jugar.

In [2]:
# prueba lunar lander por humano
# apartado A de la práctica

import gymnasium as gym

env = gym.make("LunarLander-v3", render_mode="rgb_array")

import numpy as np
import pygame
import gymnasium.utils.play

lunar_lander_keys = {
    (pygame.K_UP,): 2,
    (pygame.K_LEFT,): 1,
    (pygame.K_RIGHT,): 3,
}
#gymnasium.utils.play.play(env, zoom=3, keys_to_action=lunar_lander_keys, noop=0)

In [3]:
%%HTML
<center>
 <video 
    src="media/human_gameplay.mp4" 
    style="width: 400px; border-radius: 10px; border: 5px solid rgb(47, 221, 169); margin: auto 0;" 
    autoplay 
    muted 
    loop >
 </video>


</center>

Esta es una grabación de varios minutos jugando al juego con el teclado. Podemos ver que es bastante difícil por varios motivos:

- A veces se requieren tiempos de reacción muy rápidos.
- Solo puedes tener un motor encendido a la vez (Para usar dos tienes que alternar muy rápido).
- Solo puedes encender o apagar un motor (Si quieres menos potencia tienes que encender y apagar).

Sin embargo un agente IA no tendrá ningún inconveniente con estas dificultades así que es un juego adecuado para tratar de aplicar aprendizaje por refuerzo.

<h1 style='color:red ; text-align:center'>Apartado B</h1>

# Diseño de una estrategía personalizada.

A la hora de diseñar una estrategia personalizada lo enfocamos de la siguiente manera, definimos diferentes "modos" de comportamiento del agente con cierta lógica interna. Los que hemos diseñado son los siguientes.

- `stabilice_angle`: Devuelve una respuesta hacia el lado contrario del que esta girada la nave.
- `drift_dir`: Gira la nave en direccióín al objetivo.
- `boost`: Enciende el motor central.
- `idle`: no hace nada.

De este modo las reglas que hemos diseñado lo que generan en una *Distribución de Probabilidad* sobre los diferente modos de comportamiento. De tal manera que a la hora de ejecutar la tarea se comporta de manera estocástica pero el tickspeed es lo suficientemente alto como para que haya estabilidad. Las reglas que hemos diseañado siguen diferentes estrategias segúna la zona.

In [4]:
%%HTML
<div style="display: flex; flex-direction:row; align-items: center; justify-content: center; gap: 20px;">
    <img src="media/zones_schema.jpg" alt="drawing" width="400" style="display: flex"/>
    <ul style="display: flex; flex-direction: column; list-style-type: none;">
        <li>1 Estabilización: Prioriza stabilice_angle y boost</li>
        <li>2 Drif: Prioriza Drift al objetivo.</li>
        <li>3 Frenado: Prioriza frenado y estabilizado hasta que las condiciones sean edecuadas para meter idle.</li>
    </ul>
</div>

Las estrategías seguidas en cada zona son sencillas y se explican con comentarios en el código.

In [5]:
# agente deliberativo
# Apartado B de la práctica

# observaciones [x, y, vx, vy, ang, vang, pataiz, parader]
# acciones [nada, derecho, central, izquierdo]

# IMPORTANTE: tiene que ser creado por vosotros mismos, no vale la solución de OpenAI!

import os

def policy (observation):
    x, y, vx, vy, ang, vang, pl, pr = observation

    def stabilice_angle():
        if ang < 0:
            return 1
        else:
            return 3
        
    def drift_dir():
        dir = np.sign(x)

        if dir == 1:
            return 1
        else:
            return 3
        
    def boost():
        return 2
    
    def idle():
        return 0
    
    def get_decission(s_prob, d_prob, b_prob):

        p = np.random.uniform()

        if p < s_prob:
            return stabilice_angle()
        elif p < s_prob + d_prob:
            return drift_dir()
        elif p < s_prob + d_prob + b_prob:
            return boost()
        else:
            return idle()




    if y > 1.2: # Fase entrada (+Estabilización)
        return get_decission(0.35, 0.15, 0.45)
    elif abs(x) > 0.25: # Drift lateral
        if abs(ang) < 0.2: # Queremos drift
            return get_decission(0, 0.3, 0.6)
        else:
            return get_decission(0.4, 0, 0.6) # Estabilización fuerte (Si el drift es muy fuerte o el ángulo es grande queremos estabilizar, si no queremos mantener el drift)
    else: # Fase deceleración
        if vy < -0.2: # Si vamos muy rápido queremos estabilizar
            return get_decission(0.1, 0.1, 0.45)
        else:
            return get_decission(0, 0, 0) # Si no queremos decelerar fuerte, queremos mantener la posición y el ángulo, pero si el ángulo es grande queremos estabilizar aunque vayamos lento

In [8]:
env = gym.make("LunarLander-v3", render_mode="human")
observation, info = env.reset()

terminated = False
truncated = False

while not (terminated or truncated):
    action = policy(observation)
    observation, reward, terminated, truncated, info = env.step(action)

env.reset()
env.close()

In [9]:
%%HTML
<center>
 <video
    src="media\rules_gameplay.mp4" 
    style="width: 400px; border-radius: 10px; border: 5px solid rgb(47, 221, 169); margin: auto 0;" 
    autoplay 
    muted 
    loop >
 </video>
</center>

Hemos compilado varios vídeos en el archivo `rules_gameplay.mp4` y aquí se puede ver el comportamiento con la estrategia basada en reglas.

Comprobamos que el comportamiento es bueno, pero lógicamente se podría refinar sobre todo porque nuestra implementación depende mucho del ajuste manual de parámetros, sin embargo se pueden ver el comportamiento por zonas que queríamos conseguir, y también conseguimos varios aterrizajes exitosos.

<h1 style='color:red ; text-align:center'>Apartado C</h1>

# Neuroevolución.

## EVO_Handler.

Implementamos la clase `EVO_Handler` que se encarga de gestionar el proceso de neuroevolución. A continuación comentamos cada uno de los métodos implementados:

- `__init__`: Inicializa los parámetros de la clase y crea la población inicial de pesos para la red de neuronas.
- `evaluate_population`: Evalúa el rendimiento de cada individuo, se toman las siguientes considereciones.

    - Para cada individuo se ejecutan N episodios y se promedian las recompensas obtenidas para obtener una medida de fitness más estable. (Evitar que una estrategía que depende de la suerte produzca un individuo con un fitness altísimo).\
    - Se permite pasar una función de recompensa personalizada, pero si no se pasa se utiliza la función de recompensa por defecto del entorno.
- `sort_population`: Ordena la población de individuos según su fitness.
- `evolve`: Realiza el proceso evolutivo, permite que le pases como parámetros las funciones que deseas utilizar para la *Selection*, *Cruce* y *Mutation*.


In [10]:
from MLP import MLP

class EVO_Handler:
    def __init__(self, model, environment, population_size=100):
        self.env = environment
        self.model = model 
        self.population_size = population_size

        def generate_population():
            len_chromosome = len(self.model.to_chromosome())
            return np.random.uniform(-1, 1, (self.population_size, len_chromosome))
        
        self.population = generate_population()

    def evaluate_population(self, reward_function=None):
        fitness = np.zeros(self.population_size)

        N = 3
        for i in range(self.population_size):
            ch = self.population[i]
            self.model.from_chromosome(ch)

            for _ in range(N): # media de 5 episodios para cada individuo

                observation, info = self.env.reset()
                terminated = False
                truncated = False

                max_steps = 1000
                steps = 0

                while not (terminated or truncated) and steps < max_steps:
                    action = self.model.forward(observation)

                    observation, reward, terminated, truncated, info = self.env.step(action)

                    steps += 1

                    if reward_function:
                        fitness[i] += reward_function(observation, reward)
                    else:
                        fitness[i] += reward

                if steps >= max_steps:
                    fitness[i] -= 100 # Penalización por no terminar el episodio

        
        return fitness / N

    def sort_population(self, fitness):
        sorted_indices = np.argsort(fitness)[::-1]
        self.population = self.population[sorted_indices]
        self.fitness = fitness[sorted_indices]
        return sorted_indices

    
    def evolve(self, selection, crossover, mutation, generations=100, reward_function=None, elitism_percentage=0.3):

        best_fitness = -np.inf
        best_chromosome = None

        for gen in range(generations):
            fitness = self.evaluate_population(reward_function)
            sorted_indices = self.sort_population(fitness)

            if self.fitness[0] > best_fitness:
                best_fitness = self.fitness[0]
                best_chromosome = self.population[0]

            if gen % 10 == 0:
                print(f"Generation {gen} - Best fitness: {self.fitness[0]} - Average fitness: {np.mean(self.fitness)}")

            new_gen = []

            num_elite = int(self.population_size * elitism_percentage)
            new_gen.extend(self.population[:num_elite])

            while len(new_gen) < self.population_size:
                

                parent1 = selection(self.population, self.fitness)
                parent2 = selection(self.population, self.fitness)
                child = crossover(parent1, parent2)
                child = mutation(child)
                new_gen.append(child)

            self.population = np.array(new_gen)

        return best_chromosome, best_fitness

## Funciones de selección, cruce y mutación. (Numeros reales.)

Vamos a ver que funciones hemos implementado porque creemos que pueden funcionar bien en este problema teniendo en cuenta que los genes son números reales.

### Torunament Selection

Hemos implementado la función de selección por torneo, para intentar mantener la diversidad, manteniendo  pequeña, y por que es una función que según hemos visto suele funcionar bien en problemas de neuroevolución.

In [11]:
K = 3

def tournament_selection(population, fitness):
    selected_indices = np.random.choice(len(population), K, replace=False)
    selected_fitness = fitness[selected_indices]
    winner_index = selected_indices[np.argmax(selected_fitness)]
    return population[winner_index]

### blx_alpha_crossover

Está bien ya que genera una mezcla de los genes de los padres, sin mover bloques enteros de genes, lo cual es relevante en el contexto de una red neuronal que ya haya aprendido lo normal es que se beneficie de 'moverse' en una dirección  (Que es como funciona tradicionalmente con descenso de gradiente), y así evitamos que se peguen bloques que puedan meter ruido. Además el parámetro de alpha nos permite controlar el trade0ff exploración-explotación.

In [12]:
def blx_alpha_crossover(parent1, parent2):

    alpha = 0.5
    child = np.zeros_like(parent1)

    for i in range(len(parent1)):
        c_min = min(parent1[i], parent2[i])
        c_max = max(parent1[i], parent2[i])
        I = c_max - c_min

        lower_bound = c_min - alpha * I
        upper_bound = c_max + alpha * I

        child[i] = np.random.uniform(lower_bound, upper_bound)

    return child

### Uniform Crossover

También hemos pensado que simplemente copiar los genes de uno de los padres puede ser una buena estrategia para preservar la lógica de la red neuronal, y que la exploración venga dada por la mutación, y puede que tenga sentido en el contexto de nuestro problema, aunque puede que decelere la convergencia.

In [13]:
def uniform_crossover(parent1, parent2):

    p = np.random.uniform(0, 1)

    if p < 0.5:
        return parent1.copy()
    else:
        return parent2.copy()

### Mutación gaussiana

El algoritmo de mutación consiste en añadir a cada gen con una cierta probabilidad un valor aleatorio que sigue una distribución normal, intentamos mantener probabilidades de mutación relativamente altas y desviaciones típicas bajas, teniendo en cuenta que estamos usando elitismo. Además con una probabilidad más baja reiniciamos por completo un gen, como medida para evitar caer en mínimos locales.

In [14]:
def gaussian_mutation(chromosome, mutation_rate=0.4):

    new_chromosome = chromosome.copy()
    for i in range(len(chromosome)):
        if np.random.rand() < mutation_rate:
            new_chromosome[i] += np.random.normal(0, SIGMA)
            # reset esporádico
            if np.random.rand() < 0.02: # 2% de probabilidad de reset
                new_chromosome[i] = np.random.uniform(-1, 1)
    return new_chromosome

## Evolución con MLP

### Prueba 1. Topología [8, 6, 4].

Vamos a hacer una prueba con la red neuronal de los apuntes.

In [16]:
K = 3
SIGMA = 0.05

model = MLP([8, 6, 4])
env = gym.make("LunarLander-v3")
handler = EVO_Handler(model, env, population_size=30)
#best_chromosome, best_fitness = handler.evolve(tournament_selection, uniform_crossover, gaussian_mutation, generations=500)

### Evaluación.

Definimos una función para evaluar un indiduo que nos permite tanto visualizar un episodio nuevo como graban en un video el comportamiento del agente.

In [17]:
def evaluate_chromosome(chromosome, model, episodes=3, record_path=None):

    if record_path:
        env=gym.make("LunarLander-v3", render_mode="rgb_array")
        env = gym.wrappers.RecordVideo(env, record_path) if record_path else env
    else:
        env = gym.make("LunarLander-v3", render_mode="human")

    

    model.from_chromosome(chromosome)

    for _ in range(episodes):

        observation, info = env.reset()
        terminated = False
        truncated = False

        while not (terminated or truncated):
            action = model.forward(observation)
            observation, reward, terminated, truncated, info = env.step(action)


    env.close()

In [18]:
%%HTML
<center>
 <video
    src="media\rl_evo_1.mp4" 
    style="width: 400px; border-radius: 10px; border: 5px solid rgb(47, 221, 169); margin: auto 0;" 
    autoplay 
    muted 
    loop >
 </video>
</center>

Comprobamos que se observa aprendizaje, pero sin embargo la traza de entrenamiento no mostraba un proceso bastante inestable y estancado. Esto puede indicar underfitting así que vamos a probar a aumentar la topología de la red neuronal y a repetir el entrenamiento.|

### Prueba 2 Topología [8, 12, 6, 4].

Un poco más grande y con mas de una capa oculta.

In [22]:
K = 3
SIGMA = 0.1

model = MLP([8, 12, 6, 4])
env = gym.make("LunarLander-v3")
handler = EVO_Handler(model, env, population_size=30)
#best_chromosome, best_fitness = handler.evolve(tournament_selection, uniform_crossover, gaussian_mutation, generations=400)

Los resultados no son muy buenos, sin embargo seguimos pensando que un modelo más grande puede ser beneficioso para el problema, y tenemos algunas ideas que  implican modificar la implementación del modelo, en vez de hacerlo de esta manera, vamos a reimplementar el modelo en pytorch y discutiremos los cambios que hemos hecho en la implementación.

### Agente estocástico con PyTorch.

En esta red hemos instroducido varios cambios que creemos que pueden permitirnos trabajar con topologían mas grandes y tener aprendizaje significativo.

- Hemos introducido la función de activación *Tanh*.
- Introducimos una campa softmax al final. El forward se puede hacer en 2 modos. *Interpretando la salida  como probabilidades* o quedandonos siempre con el valor de máxima activación

Esperamos que estos cambios en la red nos permitan trabajar con topologías grandes ganando estabilidad.



In [23]:
import torch
from torch import nn
import torch.nn.functional as F
from torch.nn.utils import parameters_to_vector, vector_to_parameters

class TorchModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc = nn.Sequential(
            nn.Linear(8, 6),
            nn.ReLU(),
            nn.Linear(6, 6),
            nn.ReLU(),
            nn.Linear(6, 4)
        )

    def forward(self, x):
        x = torch.tensor(x, dtype=torch.float32)

        with torch.no_grad():
            x = self.fc(x)
            x = F.softmax(x, dim=0)
            return np.random.choice(4, p=x.numpy())
    


    def from_chromosome(self, chromosome):
        vector_to_parameters(torch.tensor(chromosome, dtype=torch.float32), self.parameters())
    
    def to_chromosome(self):
        return parameters_to_vector(self.parameters()).detach().numpy()
    
torch_model = TorchModel()

In [24]:
K = 3
SIGMA = 0.05

model = TorchModel()
env = gym.make("LunarLander-v3")
handler = EVO_Handler(model, env, population_size=40)
#best_chromosome, best_fitness = handler.evolve(tournament_selection, uniform_crossover, gaussian_mutation, generations=300)

Observamos un entrenamiento más estable, aunque más lento, podemos comprobar que el rendimiento que se alcanza es bastante bueno en este caso.

tenemos un modelo entrenado con bastantes generaciones vamos a cargarlo.

In [25]:
import pickle

path = "models/lunar_lander/lunar_lander_NE_{}.pkl"
chromosome = pickle.load(open(path.format("4"), "rb"))

In [29]:
torch_model.stochastic = True
evaluate_chromosome(chromosome, torch_model, episodes=1)

como hemos comprobado que este modelo da bastante buenos resultados.

In [30]:
%%HTML

<center>
 <video
    src="media\rl_evo_2.mp4" 
    style="width: 400px; border-radius: 10px; border: 5px solid rgb(47, 221, 169); margin: auto 0;" 
    autoplay 
    muted 
    loop >
 </video>
</center>

Como podemos observar con estas personalizaciones en la red los resultados son bastante buenos. Hemos hecho pruebas aumentando la topología de la red pero no hemos consguido mejorar los resultados, aunque es posible que aumentando drásticamente el tiempo de entrenamiento se pueda entrenar una red más grande que de mejores resultados.

## Custom Fitness.

Para el objetivo opcional de mantener la nave suspendida en el aire. La función se basa en los siguiente.

- Calcula la la distancia al objetivo y se lo pasas a un kernel RBF para obtener una recompensa basada en la cercanía al objetivo.
- Eliminas las recompensas por aterrizar o chocar.
- Penalizas salirse de la zona de juego. (mínimo local claro.)

In [31]:
def custom_reward(observation, reward):
    x, y, vx, vy, ang, vang, pl, pr = observation

    target_pos = np.array([0, 0.5])  # Posición objetivo (centro del mapa)
    current_pos = np.array([x, y])

    if reward in (-100, 100, 200) or pl or pr:
        return -50
    else:
        new_reward = 0

    if abs(x) > 1:
        new_reward -= 20

    if abs(y) > 1.5:
        new_reward -= 20

    

    distance = np.linalg.norm(current_pos - target_pos, ord=2)
    new_reward += np.exp(-distance/0.2)
    
    return new_reward

### Definamos un modelo más grande.

Vamos a probar a resolver el problema con un modelo más grande, y que además tendrá 2 ramas una para procesar las 6 primera entradas y otra (mas ligera) solo para las 2 últimas que luego se fusionarán. Esto es así por que la naturaleza de las 6 primeras son variables físicas continuas y las otras 2 son booleanos que indican si las patas han tocado el suelo, y creemos que procesar estas últimas con una rama más ligera puede ser beneficioso para el aprendizaje. 

In [32]:
class TorchModel(nn.Module):
    def __init__(self):
        super().__init__()
        # Rama para los 6 valores físicos
        self.physics_branch = nn.Sequential(
            nn.Linear(6, 12),
            nn.Tanh(),
            nn.Linear(12, 8),
            nn.Tanh(),
        )
        # Rama para los sensores de contacto de las patas
        self.contact_branch = nn.Sequential(
            nn.Linear(2, 4),
            nn.Tanh(),
        )
        # Fusión final
        self.head = nn.Linear(8 + 4, 4)

        self.deterministic = False    

    def forward(self, x):
        x = torch.tensor(x, dtype=torch.float32)
        with torch.no_grad():
            physics = self.physics_branch(x[:6])
            contact = self.contact_branch(x[6:])
            merged = torch.cat([physics, contact])
            out = F.softmax(self.head(merged), dim=0)

            if self.deterministic:
            # Si queremos una política determinista elegimos la que tenga mayor activación en la capa de salida
                return torch.argmax(out).item()
            else:
            # Tambien podemos interpretar la salida del softmax como la distribución de probabilidad de elegir cada acción, (como hacíamos en la política personlizada que definimos antes).
                return np.random.choice(4, p=out.numpy())

        return out.numpy()
    
    def from_chromosome(self, chromosome):
        vector_to_parameters(torch.tensor(chromosome, dtype=torch.float32), self.parameters())
    
    def to_chromosome(self):
        return parameters_to_vector(self.parameters()).detach().numpy()


torch_model = TorchModel()
torch_model.deterministic = False

### Entrenamiento.

In [33]:
K = 3
SIGMA = 0.1

env = gym.make("LunarLander-v3")
handler_2 = EVO_Handler(torch_model, env, population_size=30)
#best_chromosome, best_fitness = handler_2.evolve(tournament_selection, uniform_crossover, gaussian_mutation, generations=300, reward_function=custom_reward)

In [34]:
path = "models/lunar_lander/"

chromosome = pickle.load(open(path + "lunar_lander_NE_2.pkl", "rb"))

In [35]:
torch_model = TorchModel()
torch_model.deterministic = True


evaluate_chromosome(chromosome, model=torch_model, episodes=1)


En este caso observamos unos resultados prácticamente perfectos, parece que aumentar el tamaño de la red ha sido beneficioso, puede, en parte que este problema fuera algo más sencillo y por eso  ha sido posible la convergencia en valores tan buenos.